# **Training No Filter**

In [ ]:
# --- TAHAP 0: PERSIAPAN (MOUNT DRIVE & IMPORT) ---
print("Memulai Tahap 0: Persiapan...")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Import library
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam # Import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import pickle
from tqdm import tqdm
import glob
print("✅ Library berhasil di-import.")

# 3. Definisikan Konstanta
SEED = 123
print(f"Konstanta SEED diatur ke {SEED}.")

In [ ]:
import os

# --- 📁 CONFIGURATION PATHS 📁 ---

# 1. Path direktori dataset utama (Sesuaikan dengan direktori Google Drive masing-masing)
DATA_DIR = '/content/drive/MyDrive/Dataset_Preprocessing/02_Final_NoFilter'

# 2. Path untuk menyimpan model, label, dan array split
SAVE_DIR = '/content/drive/MyDrive/Output_Model/Model_NoFilter'

# 3. Daftar sampel spesifik untuk visualisasi
target_sampel_files_final = [
    'Class_A/sample_01.jpg',
    'Class_B/sample_01.jpg',
    'Class_C/sample_01.jpg',
    'Class_D/sample_01.jpg',
    'Class_E/sample_01.jpg',
    'Class_F/sample_01.jpg'
]

# --- 📁 GENERATE DERIVED PATHS 📁 ---
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'mobilenetv2_facerecog_model.h5')
LABEL_ENCODER_PATH = os.path.join(SAVE_DIR, 'label_encoder_classes.pkl')

# Path untuk menyimpan split array
X_TRAIN_PATH = os.path.join(SAVE_DIR, 'X_train.npy')
Y_TRAIN_PATH = os.path.join(SAVE_DIR, 'y_train.npy')
X_VAL_PATH = os.path.join(SAVE_DIR, 'X_val.npy')
Y_VAL_PATH = os.path.join(SAVE_DIR, 'y_val.npy')
X_TEST_PATH = os.path.join(SAVE_DIR, 'X_test.npy')
Y_TEST_PATH = os.path.join(SAVE_DIR, 'y_test.npy')

# Memastikan direktori tujuan tersedia
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Data akan dimuat dari: {DATA_DIR}")
print(f"Model/Split akan disimpan di: {SAVE_DIR}")

In [ ]:
# --- POIN 1: MEMBUAT LABEL ENCODER ---
print("\n--- Memulai Poin 1: Membuat Label Encoder ---")

try:
    class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
    num_classes = len(class_names)
    if num_classes == 0:
        print(f"ERROR: Tidak ada folder kelas ditemukan di {DATA_DIR}")
    else:
        label_encoder = LabelEncoder()
        label_encoder.fit(class_names)
        print(f"✅ Poin 1 Selesai. LabelEncoder dibuat untuk {num_classes} kelas.")
except FileNotFoundError:
    print(f"ERROR: Path data tidak ditemukan: {DATA_DIR}")

In [ ]:
# --- POIN 2: MEMUAT DAN PROSES DATA ---
print("\n--- Memulai Poin 2: Memuat Semua Data ke Memori ---")
data = []; labels = []
try:
    for class_name in tqdm(class_names, desc="Memuat Kelas"):
        class_path = os.path.join(DATA_DIR, class_name)
        image_files = []
        for ext in ('*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG'):
            image_files.extend(glob.glob(os.path.join(class_path, ext)))
        for img_path in image_files:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                data.append(img)
                labels.append(class_name)

    X = np.array(data)
    y_labels = np.array(labels)
    X = X.astype('float32') / 255.0 # Normalisasi

    print("\n✅ Poin 2 Selesai.")
    print(f"Shape X (gambar): {X.shape}") # Harusnya (2456, 224, 224, 3)
    print(f"Shape y (label): {y_labels.shape}") # Harusnya (2456,)
except Exception as e:
    print(f"\nTerjadi error saat memuat data: {e}")

In [ ]:
# --- POIN 3: MENGODEKAN LABEL ---
print("\n--- Memulai Poin 3: Mengodekan Label ---")
y_encoded = label_encoder.transform(y_labels)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)
print("✅ Poin 3 Selesai.")
print(f"Label one-hot (categorical) shape: {y_categorical.shape}") # (2456, 82)

In [ ]:
# --- POIN 4: MEMBAGI DATASET (SPLIT) & MENYIMPAN ---
print("\n--- Memulai Poin 4: Membagi Dataset ---")

if os.path.exists(X_TRAIN_PATH) and os.path.exists(X_TEST_PATH):
    print("File split .npy ditemukan. Memuat dari Drive...")
    X_train = np.load(X_TRAIN_PATH)
    y_train = np.load(Y_TRAIN_PATH)
    X_val = np.load(X_VAL_PATH)
    y_val = np.load(Y_VAL_PATH)
    X_test = np.load(X_TEST_PATH)
    y_test = np.load(Y_TEST_PATH)
    print("Dataset split berhasil dimuat dari file .npy.")
else:
    print("File split .npy tidak ditemukan. Melakukan split baru (70/20/10)...")
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_categorical, test_size=0.3, random_state=SEED, stratify=y_categorical
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(0.1/0.3), random_state=SEED, stratify=y_temp
    )
    print("Menyimpan dataset split ke file .npy...")
    np.save(X_TRAIN_PATH, X_train)
    np.save(Y_TRAIN_PATH, y_train)
    np.save(X_VAL_PATH, X_val)
    np.save(Y_VAL_PATH, y_val)
    np.save(X_TEST_PATH, X_test)
    np.save(Y_TEST_PATH, y_test)
    print("Dataset split berhasil disimpan ke Drive.")

print("\n✅ Poin 4 Selesai.")
print(f"X_train shape: {X_train.shape}") # ~1719
print(f"X_val shape  : {X_val.shape}")   # ~491
print(f"X_test shape : {X_test.shape}")  # ~246

In [ ]:
# --- POIN 5: DATA AUGMENTASI ---
print("\n--- Memulai Poin 5: Mendefinisikan ImageDataGenerator ---")
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
print("✅ Poin 5 Selesai: ImageDataGenerator dibuat.")

In [ ]:
# --- POIN 6: MENAMPILKAN 6 FOTO MAHASISWA SETELAH AUGMENTASI ---
print("\n--- Memulai Poin 6: Visualisasi Augmentasi ---")
sample_images_6 = []; sample_labels_6 = []
print(f"Memuat 6 sampel spesifik dari: {DATA_DIR}")
for file_path in target_sampel_files_final:
    full_path = os.path.join(DATA_DIR, file_path)
    img = cv2.imread(full_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sample_images_6.append(img)
        sample_labels_6.append(os.path.basename(os.path.dirname(file_path)))
    else: print(f"Peringatan: Gagal memuat sampel {full_path}")

if sample_images_6:
    images_batch = np.array(sample_images_6).astype('float32') / 255.0
    print("Menampilkan 6 sampel setelah augmentasi...")
    fig, axes = plt.subplots(2, 6, figsize=(20, 8))
    fig.suptitle('Sample 6 Foto (Setelah Augmentasi) dan Histogramnya', fontsize=16)
    augmented_batch = next(train_datagen.flow(images_batch, batch_size=6, shuffle=False))
    for i in range(len(augmented_batch)):
        img_aug = augmented_batch[i]; label = sample_labels_6[i]
        axes[0, i].imshow(img_aug); axes[0, i].set_title(label, fontsize=10); axes[0, i].axis('off')
        img_aug_hist = (img_aug * 255).astype("uint8")
        colors = ('r', 'g', 'b')
        for j, color in enumerate(colors):
            hist = cv2.calcHist([img_aug_hist], [j], None, [256], [0, 256])
            axes[1, i].plot(hist, color=color)
        axes[1, i].set_title(f'Histogram'); axes[1, i].legend(['R', 'G', 'B'], loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()
    print("✅ Poin 6 Selesai.")
else:
    print("Gagal memuat 6 gambar sampel. Poin 6 dilewati.")

In [ ]:
# --- JALANKAN ULANG SEL 8: MEMBANGUN MODEL (RESET) ---
print("\n--- Memulai Poin 7 & 8: RESET MODEL ---")

# 1. Muat base model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
# 2. Bekukan base model
base_model.trainable = False

# 3. Bangun model final
model = Sequential([
    Input(shape=(224, 224, 3)),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
], name="MobileNetV2_FaceRecog_82_Classes")

# 4. Compile dengan LR awal
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # LR Awal
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✅ Model baru (beku) telah dibuat.")
model.summary()

In [ ]:
# --- POIN 9: TRAINING MODEL ---
print("\n--- Memulai Poin 9: Training Model ---")
import time # <-- TAMBAHKAN INI

# Tentukan jumlah Epochs
EPOCHS = 30
BATCH_SIZE_TRAIN = 32

print(f"Memulai training untuk {EPOCHS} epochs...")
train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE_TRAIN, seed=SEED)

# Catat waktu mulai
start_time_train = time.time() # <-- TAMBAHKAN INI

history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE_TRAIN,
    validation_data=(X_val, y_val),
    epochs=EPOCHS
)

# Catat waktu selesai
end_time_train = time.time() # <-- TAMBAHKAN INI
training_duration = end_time_train - start_time_train # <-- TAMBAHKAN INI

print(f"✅ Poin 9 Selesai: Training selesai.")
print(f"--- Total Waktu Training: {training_duration:.2f} detik ---")

In [ ]:
# --- POIN 10: MENAMPILKAN PLOT AKURASI DAN LOSS ---
print("\n--- Memulai Poin 10: Plot Akurasi dan Loss ---")

# 'history' berasal dari sel sebelumnya (Poin 9)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc)) # Gunakan len(acc) agar aman jika training dihentikan

plt.figure(figsize=(14, 6))

# Plot Akurasi
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.show()
print("✅ Poin 10 Selesai.")

In [ ]:
# --- POIN 11: MENAMPILKAN AKURASI MODEL ---
print("\n--- Memulai Poin 11: Evaluasi Model (Data Test) ---")

print("Mengevaluasi model dengan data test...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)

# --- Menampilkan Akurasi Training Terakhir ---
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1] # <-- TAMBAHKAN INI
final_val_loss = history.history['val_loss'][-1] # <-- TAMBAHKAN INI
final_epoch_count = len(history.history['accuracy'])

print(f"\n--- Akurasi Hasil Akhir Training (Epoch {final_epoch_count}) ---")
print(f"Akurasi Training Terakhir: {final_train_acc*100:.2f}% (Loss: {final_train_loss:.4f})") # <-- Diperbarui
print(f"Akurasi Validasi Terakhir: {final_val_acc*100:.2f}% (Loss: {final_val_loss:.4f})") # <-- Diperbarui

print(f"\n--- Akurasi pada Data Test (Hold-out) ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

print(f"\n--- Efisiensi ---") # <-- TAMBAHKAN INI
print(f"Total Waktu Training: {training_duration:.2f} detik") # <-- TAMBAHKAN INI

print("✅ Poin 11 Selesai.")

In [ ]:
# --- POIN 12 & 13: CONFUSION MATRIX & CLASSIFICATION REPORT ---
print("\n--- Memulai Poin 12 & 13: Laporan Evaluasi Lengkap ---")
from sklearn.metrics import mean_squared_error # (Jika Anda masih ingin ini)

print("Memprediksi y_pred dari X_test...")
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(f"y_true_classes shape: {y_true_classes.shape}, y_pred_classes shape: {y_pred_classes.shape}")

# --- (Opsional) MSE & RMSE ---
mse = mean_squared_error(y_test, y_pred_probs)
rmse = np.sqrt(mse)

# 12. Confusion Matrix
print("\nMenampilkan Confusion Matrix...")
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix (82 Kelas)', fontsize=16)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
tick_spacing = max(1, num_classes // 20)
plt.xticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=90)
plt.yticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=0)
plt.show()
print("✅ Poin 12 Selesai.")

# 13. Classification Report
print("\nLaporan Klasifikasi (Classification Report):")
all_labels_indices = list(range(num_classes))

# --- PEMBARUAN DI SINI ---
# Minta laporan sebagai kamus (dict) untuk mengambil data
report_dict = classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True # <-- TAMBAHKAN INI
)

# Tampilkan laporan teks biasa
print(classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0
))
# -------------------------

# --- (BARU) Ekstraksi Metrik untuk Tabel Perbandingan ---
weighted_avg = report_dict['weighted avg']
precision = weighted_avg['precision']
recall = weighted_avg['recall']
f1_score = weighted_avg['f1-score']

print("\n--- Metrik Kunci untuk Tabel Perbandingan ---")
print(f"Akurasi (Test)    : {test_accuracy:.4f}")
print(f"Precision (Weighted): {precision:.4f}")
print(f"Recall (Weighted)   : {recall:.4f}")
print(f"F1-Score (Weighted) : {f1_score:.4f}")
print(f"MSE                 : {mse:.6f}")
print(f"RMSE                : {rmse:.6f}")
# ------------------------------------------------

print("✅ Poin 13 Selesai.")

In [ ]:
# --- SEL TAMBAHAN (BARU): Mengukur Waktu Inferensi ---
print("\n--- Memulai Pengukuran Waktu Inferensi ---")
import time

# Pastikan X_test ada dan tidak kosong
if 'X_test' in locals() and len(X_test) > 0:

    # 1. Pemanasan (Warm-up)
    # Jalankan prediksi pada 1 gambar untuk "memanaskan" GPU/model
    print("Melakukan pemanasan (warm-up) model...")
    _ = model.predict(X_test[0:1], verbose=0)

    # 2. Ukur Waktu
    # Ukur waktu yang dibutuhkan untuk memprediksi SELURUH data test
    num_images_test = len(X_test)
    print(f"Mengukur waktu prediksi untuk {num_images_test} gambar test...")

    start_inference = time.time()
    _ = model.predict(X_test, verbose=0, batch_size=BATCH_SIZE_TRAIN) # Gunakan batch size untuk efisiensi
    end_inference = time.time()

    total_time = end_inference - start_inference
    # Hitung waktu rata-rata per gambar (dalam milidetik)
    time_per_image_ms = (total_time / num_images_test) * 1000

    print("\n--- Metrik Efisiensi (Inferensi) ---")
    print(f"Total waktu untuk {num_images_test} prediksi: {total_time:.4f} detik")
    print(f"Waktu Inferensi Rata-rata: {time_per_image_ms:.4f} ms per gambar")

else:
    print("Variabel 'X_test' tidak ditemukan. Pengukuran inferensi dilewati.")

In [ ]:
# --- POIN 14: MENYIMPAN MODEL DAN LABEL ENCODER ---
print("\n--- Memulai Poin 14: Menyimpan Model & Label ---")

# 1. Simpan model Keras (.h5)
# Variabel 'model' ada di memori dari Poin 9
# Variabel 'MODEL_SAVE_PATH' ada di memori dari Sel 1
print(f"Menyimpan model ke: {MODEL_SAVE_PATH}")
model.save(MODEL_SAVE_PATH)
print("Model berhasil disimpan.")

# 2. Simpan 'Label Encoder' (objek LabelEncoder)
# Variabel 'label_encoder' ada di memori dari Sel 2
# Variabel 'LABEL_ENCODER_PATH' ada di memori dari Sel 1
print(f"Menyimpan LabelEncoder (objek pickle) ke: {LABEL_ENCODER_PATH}")
with open(LABEL_ENCODER_PATH, 'wb') as f:
    pickle.dump(label_encoder, f)
print("LabelEncoder berhasil disimpan.")

print("\n🎉🎉🎉 SELURUH PROSES TRAINING SELESAI 🎉🎉🎉")

# **Training Median Filter**

In [ ]:
# --- TAHAP 0: PERSIAPAN (MOUNT DRIVE & IMPORT) ---
print("Memulai Tahap 0: Persiapan...")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Import library
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam # Import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import pickle
from tqdm import tqdm
import glob
print("✅ Library berhasil di-import.")

# 3. Definisikan Konstanta
SEED = 123
print(f"Konstanta SEED diatur ke {SEED}.")

In [ ]:
import os

# --- 📁 CONFIGURATION PATHS 📁 ---

# 1. Path direktori dataset utama (Sesuaikan dengan direktori Google Drive masing-masing)
DATA_DIR = '/content/drive/MyDrive/Dataset_Preprocessing/02_Final_NoFilter'

# 2. Path untuk menyimpan model, label, dan array split
SAVE_DIR = '/content/drive/MyDrive/Output_Model/Model_NoFilter'

# 3. Daftar sampel spesifik untuk visualisasi
target_sampel_files_final = [
    'Class_A/sample_01.jpg',
    'Class_B/sample_01.jpg',
    'Class_C/sample_01.jpg',
    'Class_D/sample_01.jpg',
    'Class_E/sample_01.jpg',
    'Class_F/sample_01.jpg'
]

# --- 📁 GENERATE DERIVED PATHS 📁 ---
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'mobilenetv2_facerecog_model.h5')
LABEL_ENCODER_PATH = os.path.join(SAVE_DIR, 'label_encoder_classes.pkl')

# Path untuk menyimpan split array
X_TRAIN_PATH = os.path.join(SAVE_DIR, 'X_train.npy')
Y_TRAIN_PATH = os.path.join(SAVE_DIR, 'y_train.npy')
X_VAL_PATH = os.path.join(SAVE_DIR, 'X_val.npy')
Y_VAL_PATH = os.path.join(SAVE_DIR, 'y_val.npy')
X_TEST_PATH = os.path.join(SAVE_DIR, 'X_test.npy')
Y_TEST_PATH = os.path.join(SAVE_DIR, 'y_test.npy')

# Memastikan direktori tujuan tersedia
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Data akan dimuat dari: {DATA_DIR}")
print(f"Model/Split akan disimpan di: {SAVE_DIR}")

In [ ]:
# --- POIN 1: MEMBUAT LABEL ENCODER ---
print("\n--- Memulai Poin 1: Membuat Label Encoder ---")

try:
    class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
    num_classes = len(class_names)
    if num_classes == 0:
        print(f"ERROR: Tidak ada folder kelas ditemukan di {DATA_DIR}")
    else:
        label_encoder = LabelEncoder()
        label_encoder.fit(class_names)
        print(f"✅ Poin 1 Selesai. LabelEncoder dibuat untuk {num_classes} kelas.")
except FileNotFoundError:
    print(f"ERROR: Path data tidak ditemukan: {DATA_DIR}")

In [ ]:
# --- POIN 2: MEMUAT DAN PROSES DATA ---
print("\n--- Memulai Poin 2: Memuat Semua Data ke Memori ---")
data = []; labels = []
try:
    for class_name in tqdm(class_names, desc="Memuat Kelas"):
        class_path = os.path.join(DATA_DIR, class_name)
        image_files = []
        for ext in ('*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG'):
            image_files.extend(glob.glob(os.path.join(class_path, ext)))
        for img_path in image_files:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                data.append(img)
                labels.append(class_name)

    X = np.array(data)
    y_labels = np.array(labels)
    X = X.astype('float32') / 255.0 # Normalisasi

    print("\n✅ Poin 2 Selesai.")
    print(f"Shape X (gambar): {X.shape}") # Harusnya (2456, 224, 224, 3)
    print(f"Shape y (label): {y_labels.shape}") # Harusnya (2456,)
except Exception as e:
    print(f"\nTerjadi error saat memuat data: {e}")

In [ ]:
# --- POIN 3: MENGODEKAN LABEL ---
print("\n--- Memulai Poin 3: Mengodekan Label ---")
y_encoded = label_encoder.transform(y_labels)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)
print("✅ Poin 3 Selesai.")
print(f"Label one-hot (categorical) shape: {y_categorical.shape}") # (2456, 82)

In [ ]:
# --- POIN 4: MEMBAGI DATASET (SPLIT) & MENYIMPAN ---
print("\n--- Memulai Poin 4: Membagi Dataset ---")

if os.path.exists(X_TRAIN_PATH) and os.path.exists(X_TEST_PATH):
    print("File split .npy ditemukan. Memuat dari Drive...")
    X_train = np.load(X_TRAIN_PATH)
    y_train = np.load(Y_TRAIN_PATH)
    X_val = np.load(X_VAL_PATH)
    y_val = np.load(Y_VAL_PATH)
    X_test = np.load(X_TEST_PATH)
    y_test = np.load(Y_TEST_PATH)
    print("Dataset split berhasil dimuat dari file .npy.")
else:
    print("File split .npy tidak ditemukan. Melakukan split baru (70/20/10)...")
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_categorical, test_size=0.3, random_state=SEED, stratify=y_categorical
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(0.1/0.3), random_state=SEED, stratify=y_temp
    )
    print("Menyimpan dataset split ke file .npy...")
    np.save(X_TRAIN_PATH, X_train)
    np.save(Y_TRAIN_PATH, y_train)
    np.save(X_VAL_PATH, X_val)
    np.save(Y_VAL_PATH, y_val)
    np.save(X_TEST_PATH, X_test)
    np.save(Y_TEST_PATH, y_test)
    print("Dataset split berhasil disimpan ke Drive.")

print("\n✅ Poin 4 Selesai.")
print(f"X_train shape: {X_train.shape}") # ~1719
print(f"X_val shape  : {X_val.shape}")   # ~491
print(f"X_test shape : {X_test.shape}")  # ~246

In [ ]:
# --- POIN 5: DATA AUGMENTASI ---
print("\n--- Memulai Poin 5: Mendefinisikan ImageDataGenerator ---")
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
print("✅ Poin 5 Selesai: ImageDataGenerator dibuat.")

In [ ]:
# --- POIN 6: MENAMPILKAN 6 FOTO MAHASISWA SETELAH AUGMENTASI ---
print("\n--- Memulai Poin 6: Visualisasi Augmentasi ---")
sample_images_6 = []; sample_labels_6 = []
print(f"Memuat 6 sampel spesifik dari: {DATA_DIR}")
for file_path in target_sampel_files_final:
    full_path = os.path.join(DATA_DIR, file_path)
    img = cv2.imread(full_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sample_images_6.append(img)
        sample_labels_6.append(os.path.basename(os.path.dirname(file_path)))
    else: print(f"Peringatan: Gagal memuat sampel {full_path}")

if sample_images_6:
    images_batch = np.array(sample_images_6).astype('float32') / 255.0
    print("Menampilkan 6 sampel setelah augmentasi...")
    fig, axes = plt.subplots(2, 6, figsize=(20, 8))
    fig.suptitle('Sample 6 Foto (Setelah Augmentasi) dan Histogramnya', fontsize=16)
    augmented_batch = next(train_datagen.flow(images_batch, batch_size=6, shuffle=False))
    for i in range(len(augmented_batch)):
        img_aug = augmented_batch[i]; label = sample_labels_6[i]
        axes[0, i].imshow(img_aug); axes[0, i].set_title(label, fontsize=10); axes[0, i].axis('off')
        img_aug_hist = (img_aug * 255).astype("uint8")
        colors = ('r', 'g', 'b')
        for j, color in enumerate(colors):
            hist = cv2.calcHist([img_aug_hist], [j], None, [256], [0, 256])
            axes[1, i].plot(hist, color=color)
        axes[1, i].set_title(f'Histogram'); axes[1, i].legend(['R', 'G', 'B'], loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()
    print("✅ Poin 6 Selesai.")
else:
    print("Gagal memuat 6 gambar sampel. Poin 6 dilewati.")

In [ ]:
# --- JALANKAN ULANG SEL 8: MEMBANGUN MODEL (RESET) ---
print("\n--- Memulai Poin 7 & 8: RESET MODEL ---")

# 1. Muat base model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
# 2. Bekukan base model
base_model.trainable = False

# 3. Bangun model final
model = Sequential([
    Input(shape=(224, 224, 3)),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
], name="MobileNetV2_FaceRecog_82_Classes")

# 4. Compile dengan LR awal
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # LR Awal
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✅ Model baru (beku) telah dibuat.")
model.summary()

In [ ]:
# --- POIN 9: TRAINING MODEL ---
print("\n--- Memulai Poin 9: Training Model ---")
import time # <-- TAMBAHKAN INI

# Tentukan jumlah Epochs
EPOCHS = 30
BATCH_SIZE_TRAIN = 32

print(f"Memulai training untuk {EPOCHS} epochs...")
train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE_TRAIN, seed=SEED)

# Catat waktu mulai
start_time_train = time.time() # <-- TAMBAHKAN INI

history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE_TRAIN,
    validation_data=(X_val, y_val),
    epochs=EPOCHS
)

# Catat waktu selesai
end_time_train = time.time() # <-- TAMBAHKAN INI
training_duration = end_time_train - start_time_train # <-- TAMBAHKAN INI

print(f"✅ Poin 9 Selesai: Training selesai.")
print(f"--- Total Waktu Training: {training_duration:.2f} detik ---")

In [ ]:
# --- POIN 10: MENAMPILKAN PLOT AKURASI DAN LOSS ---
print("\n--- Memulai Poin 10: Plot Akurasi dan Loss ---")

# 'history' berasal dari sel sebelumnya (Poin 9)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc)) # Gunakan len(acc) agar aman jika training dihentikan

plt.figure(figsize=(14, 6))

# Plot Akurasi
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.show()
print("✅ Poin 10 Selesai.")

In [ ]:
# --- POIN 11: MENAMPILKAN AKURASI MODEL ---
print("\n--- Memulai Poin 11: Evaluasi Model (Data Test) ---")

print("Mengevaluasi model dengan data test...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)

# --- Menampilkan Akurasi Training Terakhir ---
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1] # <-- TAMBAHKAN INI
final_val_loss = history.history['val_loss'][-1] # <-- TAMBAHKAN INI
final_epoch_count = len(history.history['accuracy'])

print(f"\n--- Akurasi Hasil Akhir Training (Epoch {final_epoch_count}) ---")
print(f"Akurasi Training Terakhir: {final_train_acc*100:.2f}% (Loss: {final_train_loss:.4f})") # <-- Diperbarui
print(f"Akurasi Validasi Terakhir: {final_val_acc*100:.2f}% (Loss: {final_val_loss:.4f})") # <-- Diperbarui

print(f"\n--- Akurasi pada Data Test (Hold-out) ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

print(f"\n--- Efisiensi ---") # <-- TAMBAHKAN INI
print(f"Total Waktu Training: {training_duration:.2f} detik") # <-- TAMBAHKAN INI

print("✅ Poin 11 Selesai.")

In [ ]:
# --- POIN 12 & 13: CONFUSION MATRIX & CLASSIFICATION REPORT ---
print("\n--- Memulai Poin 12 & 13: Laporan Evaluasi Lengkap ---")
from sklearn.metrics import mean_squared_error # (Jika Anda masih ingin ini)

print("Memprediksi y_pred dari X_test...")
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(f"y_true_classes shape: {y_true_classes.shape}, y_pred_classes shape: {y_pred_classes.shape}")

# --- (Opsional) MSE & RMSE ---
mse = mean_squared_error(y_test, y_pred_probs)
rmse = np.sqrt(mse)

# 12. Confusion Matrix
print("\nMenampilkan Confusion Matrix...")
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix (82 Kelas)', fontsize=16)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
tick_spacing = max(1, num_classes // 20)
plt.xticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=90)
plt.yticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=0)
plt.show()
print("✅ Poin 12 Selesai.")

# 13. Classification Report
print("\nLaporan Klasifikasi (Classification Report):")
all_labels_indices = list(range(num_classes))

# --- PEMBARUAN DI SINI ---
# Minta laporan sebagai kamus (dict) untuk mengambil data
report_dict = classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True # <-- TAMBAHKAN INI
)

# Tampilkan laporan teks biasa
print(classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0
))
# -------------------------

# --- (BARU) Ekstraksi Metrik untuk Tabel Perbandingan ---
weighted_avg = report_dict['weighted avg']
precision = weighted_avg['precision']
recall = weighted_avg['recall']
f1_score = weighted_avg['f1-score']

print("\n--- Metrik Kunci untuk Tabel Perbandingan ---")
print(f"Akurasi (Test)    : {test_accuracy:.4f}")
print(f"Precision (Weighted): {precision:.4f}")
print(f"Recall (Weighted)   : {recall:.4f}")
print(f"F1-Score (Weighted) : {f1_score:.4f}")
print(f"MSE                 : {mse:.6f}")
print(f"RMSE                : {rmse:.6f}")
# ------------------------------------------------

print("✅ Poin 13 Selesai.")

In [ ]:
# --- SEL TAMBAHAN (BARU): Mengukur Waktu Inferensi ---
print("\n--- Memulai Pengukuran Waktu Inferensi ---")
import time

# Pastikan X_test ada dan tidak kosong
if 'X_test' in locals() and len(X_test) > 0:

    # 1. Pemanasan (Warm-up)
    # Jalankan prediksi pada 1 gambar untuk "memanaskan" GPU/model
    print("Melakukan pemanasan (warm-up) model...")
    _ = model.predict(X_test[0:1], verbose=0)

    # 2. Ukur Waktu
    # Ukur waktu yang dibutuhkan untuk memprediksi SELURUH data test
    num_images_test = len(X_test)
    print(f"Mengukur waktu prediksi untuk {num_images_test} gambar test...")

    start_inference = time.time()
    _ = model.predict(X_test, verbose=0, batch_size=BATCH_SIZE_TRAIN) # Gunakan batch size untuk efisiensi
    end_inference = time.time()

    total_time = end_inference - start_inference
    # Hitung waktu rata-rata per gambar (dalam milidetik)
    time_per_image_ms = (total_time / num_images_test) * 1000

    print("\n--- Metrik Efisiensi (Inferensi) ---")
    print(f"Total waktu untuk {num_images_test} prediksi: {total_time:.4f} detik")
    print(f"Waktu Inferensi Rata-rata: {time_per_image_ms:.4f} ms per gambar")

else:
    print("Variabel 'X_test' tidak ditemukan. Pengukuran inferensi dilewati.")

In [ ]:
# --- POIN 14: MENYIMPAN MODEL DAN LABEL ENCODER ---
print("\n--- Memulai Poin 14: Menyimpan Model & Label ---")

# 1. Simpan model Keras (.h5)
# Variabel 'model' ada di memori dari Poin 9
# Variabel 'MODEL_SAVE_PATH' ada di memori dari Sel 1
print(f"Menyimpan model ke: {MODEL_SAVE_PATH}")
model.save(MODEL_SAVE_PATH)
print("Model berhasil disimpan.")

# 2. Simpan 'Label Encoder' (objek LabelEncoder)
# Variabel 'label_encoder' ada di memori dari Sel 2
# Variabel 'LABEL_ENCODER_PATH' ada di memori dari Sel 1
print(f"Menyimpan LabelEncoder (objek pickle) ke: {LABEL_ENCODER_PATH}")
with open(LABEL_ENCODER_PATH, 'wb') as f:
    pickle.dump(label_encoder, f)
print("LabelEncoder berhasil disimpan.")

print("\n🎉🎉🎉 SELURUH PROSES TRAINING SELESAI 🎉🎉🎉")

# **Training Bilateral Filter**

In [ ]:
# --- TAHAP 0: PERSIAPAN (MOUNT DRIVE & IMPORT) ---
print("Memulai Tahap 0: Persiapan...")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Import library
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam # Import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import pickle
from tqdm import tqdm
import glob
print("✅ Library berhasil di-import.")

# 3. Definisikan Konstanta
SEED = 123
print(f"Konstanta SEED diatur ke {SEED}.")

In [ ]:
import os

# --- 📁 CONFIGURATION PATHS 📁 ---

# 1. Path direktori dataset utama (Sesuaikan dengan direktori Google Drive masing-masing)
DATA_DIR = '/content/drive/MyDrive/Dataset_Preprocessing/02_Final_NoFilter'

# 2. Path untuk menyimpan model, label, dan array split
SAVE_DIR = '/content/drive/MyDrive/Output_Model/Model_NoFilter'

# 3. Daftar sampel spesifik untuk visualisasi
target_sampel_files_final = [
    'Class_A/sample_01.jpg',
    'Class_B/sample_01.jpg',
    'Class_C/sample_01.jpg',
    'Class_D/sample_01.jpg',
    'Class_E/sample_01.jpg',
    'Class_F/sample_01.jpg'
]

# --- 📁 GENERATE DERIVED PATHS 📁 ---
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'mobilenetv2_facerecog_model.h5')
LABEL_ENCODER_PATH = os.path.join(SAVE_DIR, 'label_encoder_classes.pkl')

# Path untuk menyimpan split array
X_TRAIN_PATH = os.path.join(SAVE_DIR, 'X_train.npy')
Y_TRAIN_PATH = os.path.join(SAVE_DIR, 'y_train.npy')
X_VAL_PATH = os.path.join(SAVE_DIR, 'X_val.npy')
Y_VAL_PATH = os.path.join(SAVE_DIR, 'y_val.npy')
X_TEST_PATH = os.path.join(SAVE_DIR, 'X_test.npy')
Y_TEST_PATH = os.path.join(SAVE_DIR, 'y_test.npy')

# Memastikan direktori tujuan tersedia
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Data akan dimuat dari: {DATA_DIR}")
print(f"Model/Split akan disimpan di: {SAVE_DIR}")

In [ ]:
# --- POIN 1: MEMBUAT LABEL ENCODER ---
print("\n--- Memulai Poin 1: Membuat Label Encoder ---")

try:
    class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
    num_classes = len(class_names)
    if num_classes == 0:
        print(f"ERROR: Tidak ada folder kelas ditemukan di {DATA_DIR}")
    else:
        label_encoder = LabelEncoder()
        label_encoder.fit(class_names)
        print(f"✅ Poin 1 Selesai. LabelEncoder dibuat untuk {num_classes} kelas.")
except FileNotFoundError:
    print(f"ERROR: Path data tidak ditemukan: {DATA_DIR}")

In [ ]:
# --- POIN 2: MEMUAT DAN PROSES DATA ---
print("\n--- Memulai Poin 2: Memuat Semua Data ke Memori ---")
data = []; labels = []
try:
    for class_name in tqdm(class_names, desc="Memuat Kelas"):
        class_path = os.path.join(DATA_DIR, class_name)
        image_files = []
        for ext in ('*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG'):
            image_files.extend(glob.glob(os.path.join(class_path, ext)))
        for img_path in image_files:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                data.append(img)
                labels.append(class_name)

    X = np.array(data)
    y_labels = np.array(labels)
    X = X.astype('float32') / 255.0 # Normalisasi

    print("\n✅ Poin 2 Selesai.")
    print(f"Shape X (gambar): {X.shape}") # Harusnya (2456, 224, 224, 3)
    print(f"Shape y (label): {y_labels.shape}") # Harusnya (2456,)
except Exception as e:
    print(f"\nTerjadi error saat memuat data: {e}")

In [ ]:
# --- POIN 3: MENGODEKAN LABEL ---
print("\n--- Memulai Poin 3: Mengodekan Label ---")
y_encoded = label_encoder.transform(y_labels)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)
print("✅ Poin 3 Selesai.")
print(f"Label one-hot (categorical) shape: {y_categorical.shape}") # (2456, 82)

In [ ]:
# --- POIN 4: MEMBAGI DATASET (SPLIT) & MENYIMPAN ---
print("\n--- Memulai Poin 4: Membagi Dataset ---")

if os.path.exists(X_TRAIN_PATH) and os.path.exists(X_TEST_PATH):
    print("File split .npy ditemukan. Memuat dari Drive...")
    X_train = np.load(X_TRAIN_PATH)
    y_train = np.load(Y_TRAIN_PATH)
    X_val = np.load(X_VAL_PATH)
    y_val = np.load(Y_VAL_PATH)
    X_test = np.load(X_TEST_PATH)
    y_test = np.load(Y_TEST_PATH)
    print("Dataset split berhasil dimuat dari file .npy.")
else:
    print("File split .npy tidak ditemukan. Melakukan split baru (70/20/10)...")
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_categorical, test_size=0.3, random_state=SEED, stratify=y_categorical
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(0.1/0.3), random_state=SEED, stratify=y_temp
    )
    print("Menyimpan dataset split ke file .npy...")
    np.save(X_TRAIN_PATH, X_train)
    np.save(Y_TRAIN_PATH, y_train)
    np.save(X_VAL_PATH, X_val)
    np.save(Y_VAL_PATH, y_val)
    np.save(X_TEST_PATH, X_test)
    np.save(Y_TEST_PATH, y_test)
    print("Dataset split berhasil disimpan ke Drive.")

print("\n✅ Poin 4 Selesai.")
print(f"X_train shape: {X_train.shape}") # ~1719
print(f"X_val shape  : {X_val.shape}")   # ~491
print(f"X_test shape : {X_test.shape}")  # ~246

In [ ]:
# --- POIN 5: DATA AUGMENTASI ---
print("\n--- Memulai Poin 5: Mendefinisikan ImageDataGenerator ---")
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
print("✅ Poin 5 Selesai: ImageDataGenerator dibuat.")

In [ ]:
# --- POIN 6: MENAMPILKAN 6 FOTO MAHASISWA SETELAH AUGMENTASI ---
print("\n--- Memulai Poin 6: Visualisasi Augmentasi ---")
sample_images_6 = []; sample_labels_6 = []
print(f"Memuat 6 sampel spesifik dari: {DATA_DIR}")
for file_path in target_sampel_files_final:
    full_path = os.path.join(DATA_DIR, file_path)
    img = cv2.imread(full_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sample_images_6.append(img)
        sample_labels_6.append(os.path.basename(os.path.dirname(file_path)))
    else: print(f"Peringatan: Gagal memuat sampel {full_path}")

if sample_images_6:
    images_batch = np.array(sample_images_6).astype('float32') / 255.0
    print("Menampilkan 6 sampel setelah augmentasi...")
    fig, axes = plt.subplots(2, 6, figsize=(20, 8))
    fig.suptitle('Sample 6 Foto (Setelah Augmentasi) dan Histogramnya', fontsize=16)
    augmented_batch = next(train_datagen.flow(images_batch, batch_size=6, shuffle=False))
    for i in range(len(augmented_batch)):
        img_aug = augmented_batch[i]; label = sample_labels_6[i]
        axes[0, i].imshow(img_aug); axes[0, i].set_title(label, fontsize=10); axes[0, i].axis('off')
        img_aug_hist = (img_aug * 255).astype("uint8")
        colors = ('r', 'g', 'b')
        for j, color in enumerate(colors):
            hist = cv2.calcHist([img_aug_hist], [j], None, [256], [0, 256])
            axes[1, i].plot(hist, color=color)
        axes[1, i].set_title(f'Histogram'); axes[1, i].legend(['R', 'G', 'B'], loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()
    print("✅ Poin 6 Selesai.")
else:
    print("Gagal memuat 6 gambar sampel. Poin 6 dilewati.")

In [ ]:
# --- JALANKAN ULANG SEL 8: MEMBANGUN MODEL (RESET) ---
print("\n--- Memulai Poin 7 & 8: RESET MODEL ---")

# 1. Muat base model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
# 2. Bekukan base model
base_model.trainable = False

# 3. Bangun model final
model = Sequential([
    Input(shape=(224, 224, 3)),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
], name="MobileNetV2_FaceRecog_82_Classes")

# 4. Compile dengan LR awal
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # LR Awal
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✅ Model baru (beku) telah dibuat.")
model.summary()

In [ ]:
# --- POIN 9: TRAINING MODEL ---
print("\n--- Memulai Poin 9: Training Model ---")
import time # <-- TAMBAHKAN INI

# Tentukan jumlah Epochs
EPOCHS = 30
BATCH_SIZE_TRAIN = 32

print(f"Memulai training untuk {EPOCHS} epochs...")
train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE_TRAIN, seed=SEED)

# Catat waktu mulai
start_time_train = time.time() # <-- TAMBAHKAN INI

history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE_TRAIN,
    validation_data=(X_val, y_val),
    epochs=EPOCHS
)

# Catat waktu selesai
end_time_train = time.time() # <-- TAMBAHKAN INI
training_duration = end_time_train - start_time_train # <-- TAMBAHKAN INI

print(f"✅ Poin 9 Selesai: Training selesai.")
print(f"--- Total Waktu Training: {training_duration:.2f} detik ---")

In [ ]:
# --- POIN 10: MENAMPILKAN PLOT AKURASI DAN LOSS ---
print("\n--- Memulai Poin 10: Plot Akurasi dan Loss ---")

# 'history' berasal dari sel sebelumnya (Poin 9)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc)) # Gunakan len(acc) agar aman jika training dihentikan

plt.figure(figsize=(14, 6))

# Plot Akurasi
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.show()
print("✅ Poin 10 Selesai.")

In [ ]:
# --- POIN 11: MENAMPILKAN AKURASI MODEL ---
print("\n--- Memulai Poin 11: Evaluasi Model (Data Test) ---")

print("Mengevaluasi model dengan data test...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)

# --- Menampilkan Akurasi Training Terakhir ---
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1] # <-- TAMBAHKAN INI
final_val_loss = history.history['val_loss'][-1] # <-- TAMBAHKAN INI
final_epoch_count = len(history.history['accuracy'])

print(f"\n--- Akurasi Hasil Akhir Training (Epoch {final_epoch_count}) ---")
print(f"Akurasi Training Terakhir: {final_train_acc*100:.2f}% (Loss: {final_train_loss:.4f})") # <-- Diperbarui
print(f"Akurasi Validasi Terakhir: {final_val_acc*100:.2f}% (Loss: {final_val_loss:.4f})") # <-- Diperbarui

print(f"\n--- Akurasi pada Data Test (Hold-out) ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

print(f"\n--- Efisiensi ---") # <-- TAMBAHKAN INI
print(f"Total Waktu Training: {training_duration:.2f} detik") # <-- TAMBAHKAN INI

print("✅ Poin 11 Selesai.")

In [ ]:
# --- POIN 12 & 13: CONFUSION MATRIX & CLASSIFICATION REPORT ---
print("\n--- Memulai Poin 12 & 13: Laporan Evaluasi Lengkap ---")
from sklearn.metrics import mean_squared_error # (Jika Anda masih ingin ini)

print("Memprediksi y_pred dari X_test...")
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(f"y_true_classes shape: {y_true_classes.shape}, y_pred_classes shape: {y_pred_classes.shape}")

# --- (Opsional) MSE & RMSE ---
mse = mean_squared_error(y_test, y_pred_probs)
rmse = np.sqrt(mse)

# 12. Confusion Matrix
print("\nMenampilkan Confusion Matrix...")
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix (82 Kelas)', fontsize=16)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
tick_spacing = max(1, num_classes // 20)
plt.xticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=90)
plt.yticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=0)
plt.show()
print("✅ Poin 12 Selesai.")

# 13. Classification Report
print("\nLaporan Klasifikasi (Classification Report):")
all_labels_indices = list(range(num_classes))

# --- PEMBARUAN DI SINI ---
# Minta laporan sebagai kamus (dict) untuk mengambil data
report_dict = classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True # <-- TAMBAHKAN INI
)

# Tampilkan laporan teks biasa
print(classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0
))
# -------------------------

# --- (BARU) Ekstraksi Metrik untuk Tabel Perbandingan ---
weighted_avg = report_dict['weighted avg']
precision = weighted_avg['precision']
recall = weighted_avg['recall']
f1_score = weighted_avg['f1-score']

print("\n--- Metrik Kunci untuk Tabel Perbandingan ---")
print(f"Akurasi (Test)    : {test_accuracy:.4f}")
print(f"Precision (Weighted): {precision:.4f}")
print(f"Recall (Weighted)   : {recall:.4f}")
print(f"F1-Score (Weighted) : {f1_score:.4f}")
print(f"MSE                 : {mse:.6f}")
print(f"RMSE                : {rmse:.6f}")
# ------------------------------------------------

print("✅ Poin 13 Selesai.")

In [ ]:
# --- SEL TAMBAHAN (BARU): Mengukur Waktu Inferensi ---
print("\n--- Memulai Pengukuran Waktu Inferensi ---")
import time

# Pastikan X_test ada dan tidak kosong
if 'X_test' in locals() and len(X_test) > 0:

    # 1. Pemanasan (Warm-up)
    # Jalankan prediksi pada 1 gambar untuk "memanaskan" GPU/model
    print("Melakukan pemanasan (warm-up) model...")
    _ = model.predict(X_test[0:1], verbose=0)

    # 2. Ukur Waktu
    # Ukur waktu yang dibutuhkan untuk memprediksi SELURUH data test
    num_images_test = len(X_test)
    print(f"Mengukur waktu prediksi untuk {num_images_test} gambar test...")

    start_inference = time.time()
    _ = model.predict(X_test, verbose=0, batch_size=BATCH_SIZE_TRAIN) # Gunakan batch size untuk efisiensi
    end_inference = time.time()

    total_time = end_inference - start_inference
    # Hitung waktu rata-rata per gambar (dalam milidetik)
    time_per_image_ms = (total_time / num_images_test) * 1000

    print("\n--- Metrik Efisiensi (Inferensi) ---")
    print(f"Total waktu untuk {num_images_test} prediksi: {total_time:.4f} detik")
    print(f"Waktu Inferensi Rata-rata: {time_per_image_ms:.4f} ms per gambar")

else:
    print("Variabel 'X_test' tidak ditemukan. Pengukuran inferensi dilewati.")

In [ ]:
# --- POIN 14: MENYIMPAN MODEL DAN LABEL ENCODER ---
print("\n--- Memulai Poin 14: Menyimpan Model & Label ---")

# 1. Simpan model Keras (.h5)
# Variabel 'model' ada di memori dari Poin 9
# Variabel 'MODEL_SAVE_PATH' ada di memori dari Sel 1
print(f"Menyimpan model ke: {MODEL_SAVE_PATH}")
model.save(MODEL_SAVE_PATH)
print("Model berhasil disimpan.")

# 2. Simpan 'Label Encoder' (objek LabelEncoder)
# Variabel 'label_encoder' ada di memori dari Sel 2
# Variabel 'LABEL_ENCODER_PATH' ada di memori dari Sel 1
print(f"Menyimpan LabelEncoder (objek pickle) ke: {LABEL_ENCODER_PATH}")
with open(LABEL_ENCODER_PATH, 'wb') as f:
    pickle.dump(label_encoder, f)
print("LabelEncoder berhasil disimpan.")

print("\n🎉🎉🎉 SELURUH PROSES TRAINING SELESAI 🎉🎉🎉")

# **Training Gaussian Filter**

In [ ]:
# --- TAHAP 0: PERSIAPAN (MOUNT DRIVE & IMPORT) ---
print("Memulai Tahap 0: Persiapan...")

# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2. Import library
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam # Import Adam
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import cv2
import pickle
from tqdm import tqdm
import glob
print("✅ Library berhasil di-import.")

# 3. Definisikan Konstanta
SEED = 123
print(f"Konstanta SEED diatur ke {SEED}.")

In [ ]:
import os

# --- 📁 CONFIGURATION PATHS 📁 ---

# 1. Path direktori dataset utama (Sesuaikan dengan direktori Google Drive masing-masing)
DATA_DIR = '/content/drive/MyDrive/Dataset_Preprocessing/02_Final_NoFilter'

# 2. Path untuk menyimpan model, label, dan array split
SAVE_DIR = '/content/drive/MyDrive/Output_Model/Model_NoFilter'

# 3. Daftar sampel spesifik untuk visualisasi
target_sampel_files_final = [
    'Class_A/sample_01.jpg',
    'Class_B/sample_01.jpg',
    'Class_C/sample_01.jpg',
    'Class_D/sample_01.jpg',
    'Class_E/sample_01.jpg',
    'Class_F/sample_01.jpg'
]

# --- 📁 GENERATE DERIVED PATHS 📁 ---
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, 'mobilenetv2_facerecog_model.h5')
LABEL_ENCODER_PATH = os.path.join(SAVE_DIR, 'label_encoder_classes.pkl')

# Path untuk menyimpan split array
X_TRAIN_PATH = os.path.join(SAVE_DIR, 'X_train.npy')
Y_TRAIN_PATH = os.path.join(SAVE_DIR, 'y_train.npy')
X_VAL_PATH = os.path.join(SAVE_DIR, 'X_val.npy')
Y_VAL_PATH = os.path.join(SAVE_DIR, 'y_val.npy')
X_TEST_PATH = os.path.join(SAVE_DIR, 'X_test.npy')
Y_TEST_PATH = os.path.join(SAVE_DIR, 'y_test.npy')

# Memastikan direktori tujuan tersedia
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"Data akan dimuat dari: {DATA_DIR}")
print(f"Model/Split akan disimpan di: {SAVE_DIR}")

In [ ]:
# --- POIN 1: MEMBUAT LABEL ENCODER ---
print("\n--- Memulai Poin 1: Membuat Label Encoder ---")

try:
    class_names = sorted([d for d in os.listdir(DATA_DIR) if os.path.isdir(os.path.join(DATA_DIR, d))])
    num_classes = len(class_names)
    if num_classes == 0:
        print(f"ERROR: Tidak ada folder kelas ditemukan di {DATA_DIR}")
    else:
        label_encoder = LabelEncoder()
        label_encoder.fit(class_names)
        print(f"✅ Poin 1 Selesai. LabelEncoder dibuat untuk {num_classes} kelas.")
except FileNotFoundError:
    print(f"ERROR: Path data tidak ditemukan: {DATA_DIR}")

In [ ]:
# --- POIN 2: MEMUAT DAN PROSES DATA ---
print("\n--- Memulai Poin 2: Memuat Semua Data ke Memori ---")
data = []; labels = []
try:
    for class_name in tqdm(class_names, desc="Memuat Kelas"):
        class_path = os.path.join(DATA_DIR, class_name)
        image_files = []
        for ext in ('*.jpg', '*.png', '*.jpeg', '*.JPG', '*.PNG'):
            image_files.extend(glob.glob(os.path.join(class_path, ext)))
        for img_path in image_files:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                data.append(img)
                labels.append(class_name)

    X = np.array(data)
    y_labels = np.array(labels)
    X = X.astype('float32') / 255.0 # Normalisasi

    print("\n✅ Poin 2 Selesai.")
    print(f"Shape X (gambar): {X.shape}") # Harusnya (2456, 224, 224, 3)
    print(f"Shape y (label): {y_labels.shape}") # Harusnya (2456,)
except Exception as e:
    print(f"\nTerjadi error saat memuat data: {e}")

In [ ]:
# --- POIN 3: MENGODEKAN LABEL ---
print("\n--- Memulai Poin 3: Mengodekan Label ---")
y_encoded = label_encoder.transform(y_labels)
y_categorical = to_categorical(y_encoded, num_classes=num_classes)
print("✅ Poin 3 Selesai.")
print(f"Label one-hot (categorical) shape: {y_categorical.shape}") # (2456, 82)

In [ ]:
# --- POIN 4: MEMBAGI DATASET (SPLIT) & MENYIMPAN ---
print("\n--- Memulai Poin 4: Membagi Dataset ---")

if os.path.exists(X_TRAIN_PATH) and os.path.exists(X_TEST_PATH):
    print("File split .npy ditemukan. Memuat dari Drive...")
    X_train = np.load(X_TRAIN_PATH)
    y_train = np.load(Y_TRAIN_PATH)
    X_val = np.load(X_VAL_PATH)
    y_val = np.load(Y_VAL_PATH)
    X_test = np.load(X_TEST_PATH)
    y_test = np.load(Y_TEST_PATH)
    print("Dataset split berhasil dimuat dari file .npy.")
else:
    print("File split .npy tidak ditemukan. Melakukan split baru (70/20/10)...")
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y_categorical, test_size=0.3, random_state=SEED, stratify=y_categorical
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(0.1/0.3), random_state=SEED, stratify=y_temp
    )
    print("Menyimpan dataset split ke file .npy...")
    np.save(X_TRAIN_PATH, X_train)
    np.save(Y_TRAIN_PATH, y_train)
    np.save(X_VAL_PATH, X_val)
    np.save(Y_VAL_PATH, y_val)
    np.save(X_TEST_PATH, X_test)
    np.save(Y_TEST_PATH, y_test)
    print("Dataset split berhasil disimpan ke Drive.")

print("\n✅ Poin 4 Selesai.")
print(f"X_train shape: {X_train.shape}") # ~1719
print(f"X_val shape  : {X_val.shape}")   # ~491
print(f"X_test shape : {X_test.shape}")  # ~246

In [ ]:
# --- POIN 5: DATA AUGMENTASI ---
print("\n--- Memulai Poin 5: Mendefinisikan ImageDataGenerator ---")
train_datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest'
)
print("✅ Poin 5 Selesai: ImageDataGenerator dibuat.")

In [ ]:
# --- POIN 6: MENAMPILKAN 6 FOTO MAHASISWA SETELAH AUGMENTASI ---
print("\n--- Memulai Poin 6: Visualisasi Augmentasi ---")
sample_images_6 = []; sample_labels_6 = []
print(f"Memuat 6 sampel spesifik dari: {DATA_DIR}")
for file_path in target_sampel_files_final:
    full_path = os.path.join(DATA_DIR, file_path)
    img = cv2.imread(full_path)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        sample_images_6.append(img)
        sample_labels_6.append(os.path.basename(os.path.dirname(file_path)))
    else: print(f"Peringatan: Gagal memuat sampel {full_path}")

if sample_images_6:
    images_batch = np.array(sample_images_6).astype('float32') / 255.0
    print("Menampilkan 6 sampel setelah augmentasi...")
    fig, axes = plt.subplots(2, 6, figsize=(20, 8))
    fig.suptitle('Sample 6 Foto (Setelah Augmentasi) dan Histogramnya', fontsize=16)
    augmented_batch = next(train_datagen.flow(images_batch, batch_size=6, shuffle=False))
    for i in range(len(augmented_batch)):
        img_aug = augmented_batch[i]; label = sample_labels_6[i]
        axes[0, i].imshow(img_aug); axes[0, i].set_title(label, fontsize=10); axes[0, i].axis('off')
        img_aug_hist = (img_aug * 255).astype("uint8")
        colors = ('r', 'g', 'b')
        for j, color in enumerate(colors):
            hist = cv2.calcHist([img_aug_hist], [j], None, [256], [0, 256])
            axes[1, i].plot(hist, color=color)
        axes[1, i].set_title(f'Histogram'); axes[1, i].legend(['R', 'G', 'B'], loc='upper right')
    plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.show()
    print("✅ Poin 6 Selesai.")
else:
    print("Gagal memuat 6 gambar sampel. Poin 6 dilewati.")

In [ ]:
# --- JALANKAN ULANG SEL 8: MEMBANGUN MODEL (RESET) ---
print("\n--- Memulai Poin 7 & 8: RESET MODEL ---")

# 1. Muat base model
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
# 2. Bekukan base model
base_model.trainable = False

# 3. Bangun model final
model = Sequential([
    Input(shape=(224, 224, 3)),
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
], name="MobileNetV2_FaceRecog_82_Classes")

# 4. Compile dengan LR awal
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), # LR Awal
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("✅ Model baru (beku) telah dibuat.")
model.summary()

In [ ]:
# --- POIN 9: TRAINING MODEL ---
print("\n--- Memulai Poin 9: Training Model ---")
import time # <-- TAMBAHKAN INI

# Tentukan jumlah Epochs
EPOCHS = 30
BATCH_SIZE_TRAIN = 32

print(f"Memulai training untuk {EPOCHS} epochs...")
train_generator = train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE_TRAIN, seed=SEED)

# Catat waktu mulai
start_time_train = time.time() # <-- TAMBAHKAN INI

history = model.fit(
    train_generator,
    steps_per_epoch=len(X_train) // BATCH_SIZE_TRAIN,
    validation_data=(X_val, y_val),
    epochs=EPOCHS
)

# Catat waktu selesai
end_time_train = time.time() # <-- TAMBAHKAN INI
training_duration = end_time_train - start_time_train # <-- TAMBAHKAN INI

print(f"✅ Poin 9 Selesai: Training selesai.")
print(f"--- Total Waktu Training: {training_duration:.2f} detik ---")

In [ ]:
# --- POIN 10: MENAMPILKAN PLOT AKURASI DAN LOSS ---
print("\n--- Memulai Poin 10: Plot Akurasi dan Loss ---")

# 'history' berasal dari sel sebelumnya (Poin 9)
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(len(acc)) # Gunakan len(acc) agar aman jika training dihentikan

plt.figure(figsize=(14, 6))

# Plot Akurasi
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

# Plot Loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.show()
print("✅ Poin 10 Selesai.")

In [ ]:
# --- POIN 11: MENAMPILKAN AKURASI MODEL ---
print("\n--- Memulai Poin 11: Evaluasi Model (Data Test) ---")

print("Mengevaluasi model dengan data test...")
test_loss, test_accuracy = model.evaluate(X_test, y_test)

# --- Menampilkan Akurasi Training Terakhir ---
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]
final_train_loss = history.history['loss'][-1] # <-- TAMBAHKAN INI
final_val_loss = history.history['val_loss'][-1] # <-- TAMBAHKAN INI
final_epoch_count = len(history.history['accuracy'])

print(f"\n--- Akurasi Hasil Akhir Training (Epoch {final_epoch_count}) ---")
print(f"Akurasi Training Terakhir: {final_train_acc*100:.2f}% (Loss: {final_train_loss:.4f})") # <-- Diperbarui
print(f"Akurasi Validasi Terakhir: {final_val_acc*100:.2f}% (Loss: {final_val_loss:.4f})") # <-- Diperbarui

print(f"\n--- Akurasi pada Data Test (Hold-out) ---")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

print(f"\n--- Efisiensi ---") # <-- TAMBAHKAN INI
print(f"Total Waktu Training: {training_duration:.2f} detik") # <-- TAMBAHKAN INI

print("✅ Poin 11 Selesai.")

In [ ]:
# --- POIN 12 & 13: CONFUSION MATRIX & CLASSIFICATION REPORT ---
print("\n--- Memulai Poin 12 & 13: Laporan Evaluasi Lengkap ---")
from sklearn.metrics import mean_squared_error # (Jika Anda masih ingin ini)

print("Memprediksi y_pred dari X_test...")
y_pred_probs = model.predict(X_test)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(f"y_true_classes shape: {y_true_classes.shape}, y_pred_classes shape: {y_pred_classes.shape}")

# --- (Opsional) MSE & RMSE ---
mse = mean_squared_error(y_test, y_pred_probs)
rmse = np.sqrt(mse)

# 12. Confusion Matrix
print("\nMenampilkan Confusion Matrix...")
cm = confusion_matrix(y_true_classes, y_pred_classes)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix (82 Kelas)', fontsize=16)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
tick_spacing = max(1, num_classes // 20)
plt.xticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=90)
plt.yticks(ticks=np.arange(0.5, num_classes, tick_spacing), labels=[label_encoder.classes_[i] for i in range(0, num_classes, tick_spacing)], rotation=0)
plt.show()
print("✅ Poin 12 Selesai.")

# 13. Classification Report
print("\nLaporan Klasifikasi (Classification Report):")
all_labels_indices = list(range(num_classes))

# --- PEMBARUAN DI SINI ---
# Minta laporan sebagai kamus (dict) untuk mengambil data
report_dict = classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0,
    output_dict=True # <-- TAMBAHKAN INI
)

# Tampilkan laporan teks biasa
print(classification_report(
    y_true_classes,
    y_pred_classes,
    labels=all_labels_indices,
    target_names=label_encoder.classes_,
    zero_division=0
))
# -------------------------

# --- (BARU) Ekstraksi Metrik untuk Tabel Perbandingan ---
weighted_avg = report_dict['weighted avg']
precision = weighted_avg['precision']
recall = weighted_avg['recall']
f1_score = weighted_avg['f1-score']

print("\n--- Metrik Kunci untuk Tabel Perbandingan ---")
print(f"Akurasi (Test)    : {test_accuracy:.4f}")
print(f"Precision (Weighted): {precision:.4f}")
print(f"Recall (Weighted)   : {recall:.4f}")
print(f"F1-Score (Weighted) : {f1_score:.4f}")
print(f"MSE                 : {mse:.6f}")
print(f"RMSE                : {rmse:.6f}")
# ------------------------------------------------

print("✅ Poin 13 Selesai.")

In [ ]:
# --- SEL TAMBAHAN (BARU): Mengukur Waktu Inferensi ---
print("\n--- Memulai Pengukuran Waktu Inferensi ---")
import time

# Pastikan X_test ada dan tidak kosong
if 'X_test' in locals() and len(X_test) > 0:

    # 1. Pemanasan (Warm-up)
    # Jalankan prediksi pada 1 gambar untuk "memanaskan" GPU/model
    print("Melakukan pemanasan (warm-up) model...")
    _ = model.predict(X_test[0:1], verbose=0)

    # 2. Ukur Waktu
    # Ukur waktu yang dibutuhkan untuk memprediksi SELURUH data test
    num_images_test = len(X_test)
    print(f"Mengukur waktu prediksi untuk {num_images_test} gambar test...")

    start_inference = time.time()
    _ = model.predict(X_test, verbose=0, batch_size=BATCH_SIZE_TRAIN) # Gunakan batch size untuk efisiensi
    end_inference = time.time()

    total_time = end_inference - start_inference
    # Hitung waktu rata-rata per gambar (dalam milidetik)
    time_per_image_ms = (total_time / num_images_test) * 1000

    print("\n--- Metrik Efisiensi (Inferensi) ---")
    print(f"Total waktu untuk {num_images_test} prediksi: {total_time:.4f} detik")
    print(f"Waktu Inferensi Rata-rata: {time_per_image_ms:.4f} ms per gambar")

else:
    print("Variabel 'X_test' tidak ditemukan. Pengukuran inferensi dilewati.")

In [ ]:
# --- POIN 14: MENYIMPAN MODEL DAN LABEL ENCODER ---
print("\n--- Memulai Poin 14: Menyimpan Model & Label ---")

# 1. Simpan model Keras (.h5)
# Variabel 'model' ada di memori dari Poin 9
# Variabel 'MODEL_SAVE_PATH' ada di memori dari Sel 1
print(f"Menyimpan model ke: {MODEL_SAVE_PATH}")
model.save(MODEL_SAVE_PATH)
print("Model berhasil disimpan.")

# 2. Simpan 'Label Encoder' (objek LabelEncoder)
# Variabel 'label_encoder' ada di memori dari Sel 2
# Variabel 'LABEL_ENCODER_PATH' ada di memori dari Sel 1
print(f"Menyimpan LabelEncoder (objek pickle) ke: {LABEL_ENCODER_PATH}")
with open(LABEL_ENCODER_PATH, 'wb') as f:
    pickle.dump(label_encoder, f)
print("LabelEncoder berhasil disimpan.")

print("\n🎉🎉🎉 SELURUH PROSES TRAINING SELESAI 🎉🎉🎉")